In [1]:
from langchain.agents import Tool

def calculator_fn(query: str) -> str:
    return str(eval(query))  # simple example, be careful with eval in real apps

calculator_tool = Tool(
    name="Calculator",
    func=calculator_fn,
    description="Use this to solve math expressions."
)


In [2]:
from langchain.agents import initialize_agent, load_tools
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Initialize agent with custom tool
tools = [calculator_tool]

agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent="zero-shot-react-description",
    verbose=True
)

# Agent decides to call tool automatically
agent.invoke("What is 23 * 47?")


/tmp/ipykernel_10206/2805501298.py:9: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent = initialize_agent(




> Entering new AgentExecutor chain...
To find the product of 23 and 47, I will perform the multiplication.  
Action: Calculator  
Action Input: 23 * 47  
Observation: 1081
Thought:I now know the final answer.  
Final Answer: 1081

> Finished chain.


{'input': 'What is 23 * 47?', 'output': '1081'}

### Custom Tools with Multi-Agent Systems

In [3]:
def wikipedia_search(query: str) -> str:
    # Placeholder: in real app, integrate Wikipedia API
    return f"Facts about {query} from Wikipedia."

wiki_tool = Tool(
    name="WikipediaSearch",
    func=wikipedia_search,
    description="Use this to get factual information from Wikipedia."
)


In [4]:
from langchain.agents import Tool

def researcher_with_tools(state):
    topic = state.get("user_input", "AI")
    # Use tool directly
    facts = wiki_tool.run(topic)
    state["research"] = facts
    print("\n[Researcher 📚]", facts)
    return state


In [ ]:
# Just sketching the node setup
workflow.add_node("researcher", researcher_with_tools)
workflow.add_node("writer", writer)
workflow.add_node("critic", critic)


######################################################
## start


In [6]:
# Example: fetch stock prices (mock data)
def get_stock_price(ticker: str) -> str:
    # In a real scenario, call an API like Yahoo Finance
    prices = {"AAPL": 178.23, "GOOGL": 135.45, "TSLA": 254.78}
    price = prices.get(ticker.upper(), "Ticker not found")
    return f"The current price of {ticker.upper()} is {price}"


In [7]:
from langchain.agents import Tool

stock_tool = Tool(
    name="StockPriceFetcher",
    func=get_stock_price,
    description="Use this to get the current stock price for a given ticker symbol."
)


In [8]:
from langchain_openai import ChatOpenAI
from langchain.agents import initialize_agent

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

tools = [stock_tool]

agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent="zero-shot-react-description",
    verbose=True
)

# The agent automatically decides to call the tool
response = agent.invoke("What is the current price of AAPL?")
print("\nAgent Response:\n", response)




> Entering new AgentExecutor chain...
I need to find the current stock price for Apple Inc., which has the ticker symbol AAPL. 
Action: StockPriceFetcher
Action Input: "AAPL"
Observation: The current price of AAPL is 178.23
Thought:I now know the final answer
Final Answer: The current price of AAPL is 178.23.

> Finished chain.

Agent Response:
 {'input': 'What is the current price of AAPL?', 'output': 'The current price of AAPL is 178.23.'}


## Integrate into LangGraph Node

In [9]:
def researcher_node_with_tool(state):
    topic = state.get("user_input", "AAPL")
    
    # Call the stock tool
    stock_info = stock_tool.run(topic)
    
    # Agent uses returned data to generate a short summary
    llm_prompt = f"You are Researcher. The stock info is: {stock_info}\nSummarize it in 1 sentence."
    summary = llm.invoke(llm_prompt).content
    
    state["research"] = summary
    print("\n[Researcher 📊]", summary)
    return state


In [10]:
workflow = StateGraph(dict)
workflow.add_node("user", lambda state: state)
workflow.add_node("researcher", researcher_node_with_tool)
workflow.set_entry_point("user")
workflow.add_edge("user", "researcher")
workflow.add_edge("researcher", END)

state = {"user_input": "TSLA"}
app = workflow.compile()
final_state = app.invoke(state)

print("\nFinal State:", final_state)


NameError: name 'StateGraph' is not defined

In [14]:
from langchain.agents import Tool
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI

# Initialize OpenAI LLM
llm = ChatOpenAI(model="gpt-3.5-turbo")

# Example: fetch stock prices (mock data)
def get_stock_price(ticker: str) -> str:
    # In a real scenario, call an API like Yahoo Finance
    prices = {"AAPL": 178.23, "GOOGL": 135.45, "TSLA": 254.78}
    price = prices.get(ticker.upper(), "Ticker not found")
    return f"The current price of {ticker.upper()} is {price}"

stock_tool = Tool(
    name="StockPriceFetcher",
    func=get_stock_price,
    description="Use this to get the current stock price for a given ticker symbol."
)

def researcher_node_with_tool(state):
    topic = state.get("user_input", "AAPL")
    
    # Call the stock tool
    stock_info = stock_tool.run(topic)
    
    # Agent uses returned data to generate a short summary
    llm_prompt = f"You are Researcher. The stock info is: {stock_info}\nSummarize it in 1 sentence."
    summary = llm.invoke(llm_prompt)
    
    state["research"] = summary.content
    print("\n[Researcher 📊]", summary.content)
    return state

workflow = StateGraph(dict)
workflow.add_node("user", lambda state: state)
workflow.add_node("researcher", researcher_node_with_tool)
workflow.set_entry_point("user")
workflow.add_edge("user", "researcher")
workflow.add_edge("researcher", END)

state = {"user_input": "TSLA"}
app = workflow.compile()
final_state = app.invoke(state)

print("\nFinal State:", final_state)


[Researcher 📊] The current stock price of Tesla (TSLA) is $254.78.

Final State: {'user_input': 'TSLA', 'research': 'The current stock price of Tesla (TSLA) is $254.78.'}


## Full Multi-Agent Workflow with Custom Tool

In [15]:
from langchain.agents import Tool
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI

# Initialize OpenAI LLM
llm = ChatOpenAI(model="gpt-3.5-turbo")

# Example: fetch stock prices (mock data)
def get_stock_price(ticker: str) -> str:
    # In a real scenario, call an API like Yahoo Finance
    prices = {"AAPL": 178.23, "GOOGL": 135.45, "TSLA": 254.78}
    price = prices.get(ticker.upper(), "Ticker not found")
    return f"The current price of {ticker.upper()} is {price}"

stock_tool = Tool(
    name="StockPriceFetcher",
    func=get_stock_price,
    description="Use this to get the current stock price for a given ticker symbol."
)

def researcher_node_with_tool(state):
    topic = state.get("user_input", "AAPL")
    
    # Call the stock tool
    stock_info = stock_tool.run(topic)
    
    # Agent uses returned data to generate a short summary
    llm_prompt = f"You are Researcher. The stock info is: {stock_info}\nSummarize it in 1 sentence."
    summary = llm.invoke(llm_prompt)
    
    state["research"] = summary.content
    print("\n[Researcher 📊]", summary.content)
    return state

workflow = StateGraph(dict)
workflow.add_node("user", lambda state: state)
workflow.add_node("researcher", researcher_node_with_tool)
workflow.set_entry_point("user")
workflow.add_edge("user", "researcher")
workflow.add_edge("researcher", END)

state = {"user_input": "TSLA"}
app = workflow.compile()
final_state = app.invoke(state)

print("\nFinal State:", final_state)


[Researcher 📊] The current price of Tesla stock (TSLA) is 254.78.

Final State: {'user_input': 'TSLA', 'research': 'The current price of Tesla stock (TSLA) is 254.78.'}


In [22]:
from langchain.agents import Tool
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END

# Initialize LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Define Python function
def get_stock_price(ticker: str) -> str:
    prices = {"AAPL": 178.23, "GOOGL": 135.45, "TSLA": 254.78}
    price = prices.get(ticker.upper(), "Ticker not found")
    return f"The current price of {ticker.upper()} is {price}"

# Create tool instances
stock_tool = Tool(
    name="StockPriceFetcher",
    func=get_stock_price,
    description="Use this to get the current stock price for a given ticker symbol."
)

def researcher_node_with_tool(state):
    topic = state.get("user_input", "AAPL")
    
    # Call the stock tool directly
    stock_info = stock_tool.run(topic)
    
    # Agent uses returned data to generate a short summary
    llm_prompt = f"You are Researcher. The stock info is: {stock_info}\nSummarize it in 1 sentence."
    summary = llm.invoke(llm_prompt)
    
    state["research"] = summary.content
    print("\n[Researcher 📊]", summary.content)
    return state

# Create LangGraph workflow
workflow = StateGraph(dict)
workflow.add_node("user", lambda state: state)
workflow.add_node("researcher", researcher_node_with_tool)
workflow.set_entry_point("user")
workflow.add_edge("user", "researcher")
workflow.add_edge("researcher", END)

state = {"user_input": "TSLA"}
app = workflow.compile()
final_state = app.invoke(state)

print("\nFinal State:", final_state)

# For structured tools with JSON schema, use StructuredTool instead
from langchain.tools import StructuredTool

stock_tool_structured = StructuredTool.from_function(
    func=get_stock_price,
    name="StockPriceFetcher",
    description="Get the current stock price for a given ticker symbol.",
    # args_schema can be used for structured input validation
)

# Agent with structured tool
from langchain.agents import initialize_agent

tools = [stock_tool_structured]

agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent="zero-shot-react-description",
    verbose=True
)

print("\n" + "="*50)
print("AGENT WITH STRUCTURED TOOL DEMONSTRATION:")
print("="*50)
response = agent.invoke("Get the stock price of TSLA.")
print("\nAgent Response:\n", response)


[Researcher 📊] The current price of Tesla (TSLA) stock is $254.78.

Final State: {'user_input': 'TSLA', 'research': 'The current price of Tesla (TSLA) stock is $254.78.'}

AGENT WITH STRUCTURED TOOL DEMONSTRATION:


> Entering new AgentExecutor chain...
I need to fetch the current stock price for TSLA (Tesla, Inc.).  
Action: StockPriceFetcher  
Action Input: "TSLA"  
Observation: The current price of TSLA is 254.78
Thought:I now know the final answer.  
Final Answer: The current stock price of TSLA is 254.78.

> Finished chain.

Agent Response:
 {'input': 'Get the stock price of TSLA.', 'output': 'The current stock price of TSLA is 254.78.'}
